# Task 1: Web Scraping
### Evelyn Valeria Sarmiento Vásquez

In [67]:
#Instalando todas las librerías necesarias: 
#!python -m pip install selenium webdriver-manager lxml tqdm unidecode



In [68]:
#Importando las librerías que se van a necesitar:

import base64
import requests
import time
import os

import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager



In [69]:
#Abriendo la url de la UNMSM:

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
driver.maximize_window()

url = 'https://admision.unmsm.edu.pe/Website20262/A/A.html'
driver.get(url)



In [ ]:

#Creación de funciones auxiliares:
from bs4 import BeautifulSoup

#Función 1: decodifica los nombres que el sitio oculta en formato base64
def decode_obfuscated(data_auth):
    try:
        return base64.b64decode(data_auth).decode("utf-8")
    except Exception:
        return data_auth
    
#Función 2: lee todas las filas visibles en la página actual de la tabla
#Usa beautifulsoup para leer los HTML de forma rápida, recorre cada fila y decodifica

def extract_page(driver):
    """Lee las 50 filas visibles de la página actual con BeautifulSoup"""
    soup = BeautifulSoup(driver.page_source, "lxml")
    tbody = soup.select_one("#tablaPostulantes tbody")
    rows = []
    for tr in tbody.find_all("tr"):
        tds = tr.find_all("td")
        row = []
        for td in tds:
            obf = td.find(class_="obfuscated")
            if obf and obf.get("data-auth"):
                row.append(decode_obfuscated(obf["data-auth"]))
            else:
                row.append(td.get_text(strip=True))
        if row:
            rows.append(row)
    return rows


#Función 3: extrae todos los postulantes de una carrera recorriendo todas las páginas 
#Como solo se muestran 50 observaciones por vez, recorre página a página

def scrape_career(driver, url, career_name):
    driver.get(url)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "tablaPostulantes"))
    )
    time.sleep(1)

    # Preguntamos a DataTables cuántas páginas tiene esta carrera
    total_pages = driver.execute_script(
        "return $('#tablaPostulantes').DataTable().page.info().pages;"
    )

    all_rows = []
    for page_num in range(total_pages):
        if page_num > 0:
            # Guardamos el texto actual para detectar cuando cambie
            current_info = driver.find_element(By.ID, "tablaPostulantes_info").text

            # Le decimos a DataTables que vaya a la página número page_num
            driver.execute_script(
                f"$('#tablaPostulantes').DataTable().page({page_num}).draw('page');"
            )

            # Esperamos hasta que la tabla se actualice
            WebDriverWait(driver, 10).until(
                lambda d: d.find_element(By.ID, "tablaPostulantes_info").text != current_info
            )

        all_rows.extend(extract_page(driver))

    return all_rows




In [ ]:

#Extraer el link de todas las carreras 

INDEX_URL = "https://admision.unmsm.edu.pe/Website20262/A/A.html"
driver.get(INDEX_URL)

#Espera hasta que aparezcan los elementos que necesitamos "results"
WebDriverWait(driver, 15).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='results.html']"))
)

#Encontrar todas las páginas que muestren "results" y los guardamos en una lista
links = driver.find_elements(By.CSS_SELECTOR, "a[href*='results.html']")
careers = [
    (link.text.strip(), link.get_attribute("href"))
    for link in links
    if link.text.strip()
]

#Mostramos algunos links para verificar y el número de carreras
print(f"Total de carreras encontradas: {len(careers)}")
print("Ejemplo de las 5 primeras carreras:")
for name, url in tqdm(careers[:5], desc="Verificando links"):
    print(f"  {name}  →  {url}")


Total de carreras encontradas: 111
Ejemplo de las 5 primeras carreras:


Verificando links: 100%|██████████| 5/5 [00:00<00:00, 3886.49it/s]

  ADMINISTRACIÓN  →  https://admision.unmsm.edu.pe/Website20262/A/091/results.html
  ADMINISTRACIÓN - CHILCA  →  https://admision.unmsm.edu.pe/Website20262/A/0914/results.html
  ADMINISTRACIÓN - HUARAL  →  https://admision.unmsm.edu.pe/Website20262/A/0912/results.html
  ADMINISTRACIÓN - S.J.L  →  https://admision.unmsm.edu.pe/Website20262/A/0911/results.html
  ADMINISTRACIÓN - VILLA RICA  →  https://admision.unmsm.edu.pe/Website20262/A/0915/results.html


In [ ]:
#Scrapear todas las carreras

#Columnas que irán en el excel (según la página web)
COLUMNS = ["Código", "Apellidos y Nombres", "Escuela", "Puntaje", "Mérito E.P", "Observación"]

#Aquí se van a almacenar los datos
all_data = []

#Para cada carrera, extraiga todas las filas y que muestre el progreso, carrera y número de postulantes (para verificar que no solo se quede con las primeras 50 filas)
for career_name, url in tqdm(careers, desc="Scrapeando carreras"):
    tqdm.write(f"  → {career_name}")
    try:
        rows = scrape_career(driver, url, career_name)
        all_data.extend(rows)
        tqdm.write(f"     ✓ {len(rows)} postulantes")
    except Exception as e:
        tqdm.write(f"     ERROR: {e}")

print(f"\nTotal de filas recolectadas: {len(all_data)}")



Scrapeando carreras:   0%|          | 0/111 [00:00<?, ?it/s]

  → ADMINISTRACIÓN


Scrapeando carreras:   1%|          | 1/111 [00:05<09:35,  5.23s/it]

     ✓ 553 postulantes
  → ADMINISTRACIÓN - CHILCA


Scrapeando carreras:   2%|▏         | 2/111 [00:06<05:32,  3.05s/it]

...

     ✓ 144 postulantes
  → TOXICOLOGÍA


Scrapeando carreras:  99%|█████████▉| 110/111 [06:29<00:02,  2.28s/it]

     ✓ 35 postulantes
  → TRABAJO SOCIAL


Scrapeando carreras: 100%|██████████| 111/111 [06:31<00:00,  3.53s/it]

     ✓ 174 postulantes

Total de filas recolectadas: 25880


In [ ]:
#Guardar en Excel

#Crear la carpeta de output
os.makedirs("output", exist_ok=True)
OUTPUT_PATH = os.path.join("output", "resultados_sanmarcos.xlsx")

#Convertir los datos a dataframe y exportar a excel 
df = pd.DataFrame(all_data, columns=COLUMNS)
df.to_excel(OUTPUT_PATH, index=False)

print(f"Archivo guardado: {OUTPUT_PATH}")
print(f"Total filas: {len(df)}")
df.head()



Archivo guardado: output\resultados_sanmarcos.xlsx
Total filas: 25880


,Código,Apellidos y Nombres,Escuela,Puntaje,Mérito E.P,Observación
0,656550,"ABAD SALGADO, SOFIA ESTRELLA",ADMINISTRACIÓN,,,
1,565808,"ACUÑA ECHEBARRIA, YAMILA SORAYA",ADMINISTRACIÓN,,,ALCANZÓ VACANTE
2,658780,"ACUÑA PACOTAIPE, DULCE MARIA",ADMINISTRACIÓN,,,
3,651014,"AGUERO VIDARTE, ADRIAN SEBASTIAN",ADMINISTRACIÓN,,,Articulo N° 5 del Reglamento de Admisión 2026-II
4,658610,"AGUIRRE RAMOS, JOGAN ALYAIR",ADMINISTRACIÓN,,,


In [74]:
#Cerrar el navegador
driver.quit()
